In [11]:
import random
import itertools

import transformers
import torch
import datasets
import plotly.express
import einops
import tqdm.auto
from lovely_tensors import lovely

In [12]:
model_ckpt = "meta-llama/Llama-3.2-1B"
model = transformers.AutoModelForCausalLM.from_pretrained(model_ckpt).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(model_ckpt)

In [13]:
ds = datasets.concatenate_datasets(
    [
        datasets.load_dataset("RealTimeData/bbc_news_alltime", f"2024-{i:02d}", split="train").select_columns("content") for i in range(1, 13)
    ]
)

In [14]:
texts = list(set(ds["content"]))
texts.sort(key=lambda x: len(x), reverse=True)

In [15]:
tokenized = tokenizer(texts)

In [16]:
desired_length = 512
ds = [inp[:desired_length] for inp in tokenized.input_ids if len(inp) >= desired_length]
random.Random(0).shuffle(ds)
eval_size = 1000
train_ds, valid_ds, test_ds = torch.tensor(ds, dtype=torch.long).tensor_split([-eval_size*2, -eval_size])
train_ds.shape, valid_ds.shape, test_ds.shape

(torch.Size([11778, 512]), torch.Size([1000, 512]), torch.Size([1000, 512]))

In [17]:
device = "cuda:7"
dtype = torch.bfloat16
model.to(device, dtype).eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [32]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [39]:
max_offset = 3
probes = []
optims = []

# layer_idcs = range(len(model.model.layers) + 1)
layer_idcs = range(3)
offsets = range(max_offset)

for _ in layer_idcs:
    probes.append([])
    optims.append([])
    for _ in offsets:
        probe = torch.nn.Linear(model.config.hidden_size, model.config.hidden_size, bias=False, device=model.device, dtype=model.dtype)
        optim = torch.optim.AdamW(probe.parameters(), lr=1e-4)
        probes[-1].append(probe)
        optims[-1].append(optim)

In [ ]:
batch_size = 16

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(train_ds),
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    pin_memory_device=device
)
valid_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(valid_ds),
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True,
    pin_memory_device=device
)

batches = itertools.cycle(train_loader)
pbar = tqdm.auto.tqdm(batches, desc="Training")

n_valid_batches = len(valid_loader)

train_accs = [[[] for _ in offsets] for _ in layer_idcs]
valid_accs = [[[] for _ in offsets] for _ in layer_idcs]

for train_step, train_batch in enumerate(pbar):
    train_batch, = train_batch # loader outputs tuples even if there is only one x without y, we need to unpack
    train_batch = train_batch.to(device)

    with torch.no_grad():
        hidden_states = model(train_batch, output_hidden_states=True).hidden_states

    curr_train_accs = torch.zeros((len(layer_idcs), len(offsets)), device=device, dtype=torch.float32)
    for layer_idx, offset in itertools.product(layer_idcs, offsets):
        probe = probes[layer_idx][offset]
        optim = optims[layer_idx][offset]

        probe.train()
        prediction: torch.Tensor = probe(hidden_states[layer_idx])
        logits = model.lm_head(prediction)
        logits = einops.rearrange(logits, "batch seq vocab -> batch vocab seq")
        logits = logits[..., offset:] # slice from start
        labels = train_batch[:, :logits.shape[-1]] # slice from end
        optim.zero_grad()
        loss = torch.nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optim.step()
        curr_train_accs[layer_idx][offset] = (logits.argmax(dim=1) == labels).float().mean()

    if train_step % 100 == 0 and train_step != 0:
        probe.eval()
        with torch.no_grad():
            curr_valid_accs = torch.zeros((len(layer_idcs), len(offsets), n_valid_batches), device=device, dtype=torch.float32)
            for val_batch_idx, valid_batch in enumerate(tqdm.auto.tqdm(valid_loader, desc="Validating", leave=False)):
                valid_batch, = valid_batch # loader outputs tuples even if there is only one x without y, we need to unpack
                valid_batch = valid_batch.to(device)
                hidden_states = model(valid_batch, output_hidden_states=True).hidden_states

                for layer_idx, offset in itertools.product(layer_idcs, offsets):
                    prediction = probe(hidden_states[layer_idx])
                    logits = model.lm_head(prediction)
                    logits = einops.rearrange(logits, "batch seq vocab -> batch vocab seq")
                    logits = logits[..., offset:] # slice from start
                    labels = valid_batch[:, :logits.shape[-1]] # slice from end
                    valid_acc_batch = (logits.argmax(dim=1) == labels).float().mean()
                    curr_valid_accs[layer_idx, offset, val_batch_idx] = valid_acc_batch
            curr_valid_accs = curr_valid_accs.mean(dim=-1) # average over all valid batches

        for layer_idx, offset in itertools.product(layer_idcs, offsets):
            train_accs[layer_idx][offset].append(curr_train_accs[layer_idx][offset].item())
            valid_accs[layer_idx][offset].append(curr_valid_accs[layer_idx][offset].item())

        print(f"{train_step=}")
        for offset in offsets:
            # find best performing layer and timestep
            accs_tensor = torch.tensor(valid_accs)[:, offset, :]
            best_layer, best_step = divmod(accs_tensor.argmax().item(), accs_tensor.shape[-1])
            best_valid_acc = valid_accs[best_layer][offset][best_step]
            train_acc = train_accs[best_layer][offset][best_step]
            print(f"  For offset: {offset} - best layer: {best_layer:<2} from step: {best_step:<5} - valid acc: {best_valid_acc:.3f} and train acc: {train_acc:.3f}")


Training: 0it [00:00, ?it/s]

Validating:   0%|          | 0/63 [00:00<?, ?it/s]

train_step=100
  Best layer: 2  from step: 0     - valid acc: 0.0330287404358387 and train acc: 0.12353515625
  Best layer: 1  from step: 0     - valid acc: 0.034603796899318695 and train acc: 0.04721134901046753
  Best layer: 1  from step: 0     - valid acc: 0.040163010358810425 and train acc: 0.039338238537311554


Validating:   0%|          | 0/63 [00:00<?, ?it/s]

train_step=200
  Best layer: 2  from step: 0     - valid acc: 0.0330287404358387 and train acc: 0.12353515625
  Best layer: 1  from step: 1     - valid acc: 0.03691602498292923 and train acc: 0.07326320558786392
  Best layer: 2  from step: 1     - valid acc: 0.06346483528614044 and train acc: 0.06924019753932953


...

Validating:   0%|          | 0/63 [00:00<?, ?it/s]

train_step=3500
  Best layer: 2  from step: 0     - valid acc: 0.0330287404358387 and train acc: 0.12353515625
  Best layer: 2  from step: 34    - valid acc: 0.09056705981492996 and train acc: 0.2045009732246399
  Best layer: 2  from step: 34    - valid acc: 0.11079210788011551 and train acc: 0.1095588281750679


Validating:   0%|          | 0/63 [00:00<?, ?it/s]

train_step=3600
  Best layer: 2  from step: 0     - valid acc: 0.0330287404358387 and train acc: 0.12353515625
  Best layer: 2  from step: 34    - valid acc: 0.09056705981492996 and train acc: 0.2045009732246399
  Best layer: 2  from step: 35    - valid acc: 0.11090103536844254 and train acc: 0.10894608497619629


KeyboardInterrupt: 

# Stuff Bellow
Code below might be useful for the experimenting on the number data

In [ ]:
def fmt_num(seq: list[int]) -> str:
    fst, *rest = seq
    return str(fst) + "".join(f"{x:03d}" for x in rest)

rng = random.Random(0)
nums = [[rng.randint(0, 999) for _ in range(10)] for _ in range(100)]
nums_input = tokenizer([fmt_num(seq) for seq in nums], return_tensors="pt").input_ids
nums_input

In [ ]:
with torch.no_grad():
    hidden_states = model(nums_input.to(model.device), output_hidden_states=True).hidden_states
    prediction = probe(hidden_states[layer_idx])
    logits = model.lm_head(prediction)
    logits = logits[:, max_offset:, :]
    logits = einops.rearrange(logits, "b s v -> b v s")
    labels = nums_input[:, max_offset-offset:nums_input.shape[1]-offset] # shifted to predict the past token
    print(logits.argmax(dim=1).shape, labels.shape)
    valid_acc_batch = (logits.argmax(dim=1).cpu() == labels).float().mean()
    print(valid_acc_batch)

In [ ]:
tokenizer.batch_decode(logits.argmax(dim=1))